# Workshop: Machine Learning for Aquatic Remote Sensing
**Event:** International Science Council (ISC) SCOR Workshop on Satellite Remote Sensing <br>
**Location:** Department of Marine Sciences, Berhampur University, India <br>
**Date:** Dec 2025 <br>
**Instructor:** Chintan B. Maniyar, PhD Candidate, University of Georgia (chintanmaniyar@uga.edu) <br>

---
**Note on Usage:**
This notebook was developed specifically for educational purposes within the SCOR workshop curriculum. The code and workflows demonstrate the application of Machine Learning to aquatic remote sensing data. While compliant with scientific best practices, users should rigorously validate these models before applying them to operational or published research.

### Import Packages

In [ ]:
%load_ext autoreload

In [ ]:
%autoreload 2

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
import contextily as ctx
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import seaborn as sns
import joblib

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

### Reading Sentinel-3 OLCI Image for Lake Erie (Atmospherically corrected with ACOLITE)

In [ ]:
# Define the path to your unzipped .SEN3 folder
file_path = './../data/S3B_OLCI_2024_08_13_16_03_22_L2R.nc'
s3_ds = xr.open_dataset(file_path)

In [ ]:
# promote lat and long as dimensions
s3_ds = s3_ds.set_coords(['lon', 'lat'])
s3_ds

This is a full scene image, let's crop it to match the bounding box of Lake Erie's western basin.

In [ ]:
north = 42.25
south = 41.3
west = -83.6
east = -82.5

In [ ]:
# Create and apply the geographic mask
mask = (s3_ds.lat >= south) & (s3_ds.lat <= north) & (s3_ds.lon >= west) & (s3_ds.lon <= east)
s3_ds = s3_ds.where(mask, drop=True)

In [ ]:
# get all rrs bands as dimensions
rrs_band_names = [var for var in s3_ds.data_vars if var.startswith('rhos_')]
rrs_bands = [s3_ds[name] for name in rrs_band_names]
band_wavelengths = [int(name.split('_')[-1]) for name in rrs_band_names]

s3_rrs = xr.concat(rrs_bands, dim='band').assign_coords(band=band_wavelengths)
s3_rrs = s3_rrs.transpose('y', 'x', 'band')
s3_rrs

Now, let's apply a land and cloud mask to the image, by thresholding the SWIR band.

In [ ]:
plt.hist(s3_rrs.sel(band=1016, method='nearest').values.flatten())

In [ ]:
wave = 1016
threshold = 0.15 # swir threshold. Play with this to refine the mask
is_water_mask = s3_rrs.sel(band=1016, method='nearest') < threshold
s3_rrs = s3_rrs.where(is_water_mask)

### Preliminary Plot - Sanity Check

In [ ]:
# --- Plotting with Cartopy ---
# Select the data to plot (we'll just plot the red band)
data_to_plot = s3_rrs.sel(band=620, method='nearest')

fig = plt.figure(figsize=(12, 8))
ax = plt.axes(projection=ccrs.PlateCarree())

# Add geographic features
ax.add_feature(cfeature.LAND.with_scale('10m'), facecolor='lightgray')
ax.add_feature(cfeature.LAKES.with_scale('10m'), facecolor='none', edgecolor='black')
ax.coastlines(resolution='10m')

# Plot the Sentinel-3 data
data_to_plot.plot(
    ax=ax,
    x='lon',
    y='lat',
    transform=ccrs.PlateCarree(),
    cmap='viridis',
    robust=True
)

# Finalize the map
ax.gridlines(draw_labels=True, linestyle='--')
ax.set_extent([west, east, south, north])
# plt.title('Sentinel-3 Radiance over Green Bay')
plt.show()

### Read All Bands

In [ ]:
s3_rrs.isel(band=slice(1,12))

In [ ]:
s3_rrs.isel(band=slice(1,12)).values.shape

As a part of making the data model-ready, we must convert the satellite Surface Reflectance (SR) to $R{rs}$, which can be achieved by dividing the SR by $\pi$.

In [ ]:
rrs = s3_rrs.isel(band=slice(1,12)).values.reshape(-1, 11)
rrs = rrs/np.pi
rrs.shape

We create a valid mask to *remember* the locations of which pixels contain values and which were masked.

In [ ]:
valid_mask = ~np.isnan(rrs).any(axis=1)
# valid_mask = (rrs >= 0).all(axis=1)
valid_data = rrs[valid_mask]

In [ ]:
valid_data.shape

> We converted our image to a 1 dimensional list of pixels. Recall "tabular" machine learning from our last session. We are applying the model individuallly to each pixel, without worrying about the surrounding pixels at the time.

## Get Inferences from Various Models we trained

### Linear Model

In [ ]:
model = joblib.load('./../models/linear_model_rrs.joblib')

In [ ]:
y_preds = model.predict(valid_data)

In [ ]:
full_predictions = np.full(s3_rrs.shape[0]*s3_rrs.shape[1], np.nan)  # (lat*lon)
full_predictions[valid_mask] = y_preds.ravel()   
img_pred = full_predictions.reshape(s3_rrs.shape[0], s3_rrs.shape[1])

# --- Create the new DataArray for your predictions ---
predictions_xr = xr.DataArray(
    data=img_pred,
    dims=('y', 'x'),
    coords={
        'lon': s3_rrs.coords['lon'],
        'lat': s3_rrs.coords['lat']
    },
    name='Chl-a $(mg/m^3)$'  # Giving it a name is good practice
)

In [ ]:
# Manually define the Web Mercator projection used by basemap tiles
web_mercator_proj = ccrs.epsg(3857)

# --- 1. Create a figure and axes using the required Mercator projection ---
fig = plt.figure(figsize=(12, 10))
ax = plt.axes(projection=web_mercator_proj)

# --- 2. Add geographic features (optional, as basemap replaces them) ---
# For a satellite basemap, borders or states with a light color work well.
# ax.add_feature(cfeature.STATES.with_scale('10m'), edgecolor='white', linewidth=1.5, facecolor='none')

# --- 3. Plot your prediction data ---
predictions_xr.plot(
    ax=ax,
    x='lon',
    y='lat',
    transform=ccrs.PlateCarree(), # Tell cartopy your data's native projection
    cmap='jet',               # Changed from 'jet' for better perception
    robust=True,
    add_colorbar=True,
    vmin=0,
    vmax=40
    # alpha=0.7                     # Make data transparent to see basemap
)

# --- 4. Finalize the map's view ---
# Set the extent using your lat/lon box, specifying its CRS
ax.set_extent([west, east, south, north], crs=ccrs.PlateCarree())

# Add and format gridline labels to show degrees
gl = ax.gridlines(draw_labels=True, linewidth=1, color='white', alpha=0.5, linestyle='--')
gl.xformatter = LongitudeFormatter()
gl.yformatter = LatitudeFormatter()
gl.top_labels = False
gl.right_labels = False
gl.left_labels=False
gl.bottom_labels=False

# --- 5. Add the basemap ---
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, zoom='auto')

# plt.savefig('./s3_erie_pc_v2.png', dpi=900)
plt.show()

### Random Forest

In [ ]:
model = joblib.load('./../models/random_forest_rrs.joblib')

In [ ]:
y_preds = model.predict(valid_data)

In [ ]:
full_predictions = np.full(s3_rrs.shape[0]*s3_rrs.shape[1], np.nan)  # (lat*lon)
full_predictions[valid_mask] = y_preds.ravel()   
img_pred = full_predictions.reshape(s3_rrs.shape[0], s3_rrs.shape[1])

# --- Create the new DataArray for your predictions ---
predictions_xr = xr.DataArray(
    data=img_pred,
    dims=('y', 'x'),
    coords={
        'lon': s3_rrs.coords['lon'],
        'lat': s3_rrs.coords['lat']
    },
    name='Chl-a $(mg/m^3)$'  # Giving it a name is good practice
)

In [ ]:
# Manually define the Web Mercator projection used by basemap tiles
web_mercator_proj = ccrs.epsg(3857)

# --- 1. Create a figure and axes using the required Mercator projection ---
fig = plt.figure(figsize=(12, 10))
ax = plt.axes(projection=web_mercator_proj)

# --- 2. Add geographic features (optional, as basemap replaces them) ---
# For a satellite basemap, borders or states with a light color work well.
# ax.add_feature(cfeature.STATES.with_scale('10m'), edgecolor='white', linewidth=1.5, facecolor='none')

# --- 3. Plot your prediction data ---
predictions_xr.plot(
    ax=ax,
    x='lon',
    y='lat',
    transform=ccrs.PlateCarree(), # Tell cartopy your data's native projection
    cmap='jet',               # Changed from 'jet' for better perception
    robust=True,
    add_colorbar=True,
    vmin=0,
    vmax=40
    # alpha=0.7                     # Make data transparent to see basemap
)

# --- 4. Finalize the map's view ---
# Set the extent using your lat/lon box, specifying its CRS
ax.set_extent([west, east, south, north], crs=ccrs.PlateCarree())

# Add and format gridline labels to show degrees
gl = ax.gridlines(draw_labels=True, linewidth=1, color='white', alpha=0.5, linestyle='--')
gl.xformatter = LongitudeFormatter()
gl.yformatter = LatitudeFormatter()
gl.top_labels = False
gl.right_labels = False
gl.left_labels=False
gl.bottom_labels=False

# --- 5. Add the basemap ---
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, zoom='auto')

# plt.savefig('./s3_erie_pc_v2.png', dpi=900)
plt.show()

### XGB

In [ ]:
model = joblib.load('./../models/xgb_rrs.joblib')

In [ ]:
y_preds = model.predict(valid_data)

In [ ]:
full_predictions = np.full(s3_rrs.shape[0]*s3_rrs.shape[1], np.nan)  # (lat*lon)
full_predictions[valid_mask] = y_preds.ravel()   
img_pred = full_predictions.reshape(s3_rrs.shape[0], s3_rrs.shape[1])

# --- Create the new DataArray for your predictions ---
predictions_xr = xr.DataArray(
    data=img_pred,
    dims=('y', 'x'),
    coords={
        'lon': s3_rrs.coords['lon'],
        'lat': s3_rrs.coords['lat']
    },
    name='Chl-a $(mg/m^3)$'  # Giving it a name is good practice
)

In [ ]:
# Manually define the Web Mercator projection used by basemap tiles
web_mercator_proj = ccrs.epsg(3857)

# --- 1. Create a figure and axes using the required Mercator projection ---
fig = plt.figure(figsize=(12, 10))
ax = plt.axes(projection=web_mercator_proj)

# --- 2. Add geographic features (optional, as basemap replaces them) ---
# For a satellite basemap, borders or states with a light color work well.
# ax.add_feature(cfeature.STATES.with_scale('10m'), edgecolor='white', linewidth=1.5, facecolor='none')

# --- 3. Plot your prediction data ---
predictions_xr.plot(
    ax=ax,
    x='lon',
    y='lat',
    transform=ccrs.PlateCarree(), # Tell cartopy your data's native projection
    cmap='jet',               # Changed from 'jet' for better perception
    robust=True,
    add_colorbar=True,
    vmin=0,
    vmax=40
    # alpha=0.7                     # Make data transparent to see basemap
)

# --- 4. Finalize the map's view ---
# Set the extent using your lat/lon box, specifying its CRS
ax.set_extent([west, east, south, north], crs=ccrs.PlateCarree())

# Add and format gridline labels to show degrees
gl = ax.gridlines(draw_labels=True, linewidth=1, color='white', alpha=0.5, linestyle='--')
gl.xformatter = LongitudeFormatter()
gl.yformatter = LatitudeFormatter()
gl.top_labels = False
gl.right_labels = False
gl.left_labels=False
gl.bottom_labels=False

# --- 5. Add the basemap ---
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, zoom='auto')

# plt.savefig('./s3_erie_pc_v2.png', dpi=900)
plt.show()

### SVM

In [ ]:
model = joblib.load('./../models/svm_rrs.joblib')

In [ ]:
y_preds = model.predict(valid_data)

In [ ]:
full_predictions = np.full(s3_rrs.shape[0]*s3_rrs.shape[1], np.nan)  # (lat*lon)
full_predictions[valid_mask] = y_preds.ravel()   
img_pred = full_predictions.reshape(s3_rrs.shape[0], s3_rrs.shape[1])

# --- Create the new DataArray for your predictions ---
predictions_xr = xr.DataArray(
    data=img_pred,
    dims=('y', 'x'),
    coords={
        'lon': s3_rrs.coords['lon'],
        'lat': s3_rrs.coords['lat']
    },
    name='Chl-a $(mg/m^3)$'  # Giving it a name is good practice
)

In [ ]:
# Manually define the Web Mercator projection used by basemap tiles
web_mercator_proj = ccrs.epsg(3857)

# --- 1. Create a figure and axes using the required Mercator projection ---
fig = plt.figure(figsize=(12, 10))
ax = plt.axes(projection=web_mercator_proj)

# --- 2. Add geographic features (optional, as basemap replaces them) ---
# For a satellite basemap, borders or states with a light color work well.
# ax.add_feature(cfeature.STATES.with_scale('10m'), edgecolor='white', linewidth=1.5, facecolor='none')

# --- 3. Plot your prediction data ---
predictions_xr.plot(
    ax=ax,
    x='lon',
    y='lat',
    transform=ccrs.PlateCarree(), # Tell cartopy your data's native projection
    cmap='jet',               # Changed from 'jet' for better perception
    robust=True,
    add_colorbar=True,
    vmin=0,
    vmax=40
    # alpha=0.7                     # Make data transparent to see basemap
)

# --- 4. Finalize the map's view ---
# Set the extent using your lat/lon box, specifying its CRS
ax.set_extent([west, east, south, north], crs=ccrs.PlateCarree())

# Add and format gridline labels to show degrees
gl = ax.gridlines(draw_labels=True, linewidth=1, color='white', alpha=0.5, linestyle='--')
gl.xformatter = LongitudeFormatter()
gl.yformatter = LatitudeFormatter()
gl.top_labels = False
gl.right_labels = False
gl.left_labels=False
gl.bottom_labels=False

# --- 5. Add the basemap ---
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, zoom='auto')

# plt.savefig('./s3_erie_pc_v2.png', dpi=900)
plt.show()

> ### *Question*: Can you do a similar model application for Neural Networks that we trained in the 03 notebook? You can explore how to take help from an LLM for this part, and use the code above too!

In [ ]:
## your code here